# AIGC 5005 · Module 01 · Part 2
## Your Python environment

Companion notebook to Part 2 of the Module 01 learning content. Every example from that part is here, ready to run, with comments explaining what to notice.

**Estimated time:** 30 minutes

**How to use it:** read each explanation, run the cell beneath it, and compare the output with what you expected *before* you ran it. The prediction is where the learning happens. Change values and run cells again; you cannot break anything.

**Kernel:** select the kernel from your `module01-tool` conda environment (top right in VS Code or Jupyter). To make one: `conda create -n module01-tool python=3.11 ipykernel`, then `conda activate module01-tool`. If VS Code offers to install `ipykernel`, accept.

## 0. Set up a scratch folder

This notebook creates a few small files so you can inspect them. Everything goes inside a folder called `scratch_part2` next to this notebook. The last cell deletes it.

In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

SCRATCH = Path("scratch_part2").resolve()
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)           # start clean every time you re-run the notebook
SCRATCH.mkdir()
print("Working in:", SCRATCH)

## 1. Which Python is running this notebook?

The most common environment problem is a package that is installed, but not in the Python you are actually using. The first step is always to find out which Python that is.

In [ ]:
print("Python version :", platform.python_version())
print("Interpreter    :", sys.executable)      # the exact python program running this notebook
print("Environment    :", sys.prefix)          # the folder that environment lives in

### Is it a conda environment?

Every conda environment contains a `conda-meta` folder, so its presence is a reliable test. The function reports what it finds.

**Predict first:** are you in `base`, in a project environment, or outside conda altogether?

In [ ]:
def describe_environment(prefix=None, conda_name=None):
    """Report which Python environment is running this code, and whether it is a conda environment."""
    prefix = Path(prefix or sys.prefix)
    is_conda = (prefix / "conda-meta").is_dir()
    name = conda_name or os.environ.get("CONDA_DEFAULT_ENV") or (prefix.name if is_conda else None)
    print("Interpreter          :", sys.executable)
    print("Environment folder   :", prefix)
    print("Conda environment?   :", is_conda)
    if not is_conda:
        print("-> Not a conda environment. Select your conda kernel (top right) before continuing.")
    elif name == "base":
        print("-> You are in base. Create a project environment instead:  conda create -n module01-tool python=3.11 ipykernel")
    else:
        print(f"-> Good: working in the '{name}' environment.")


describe_environment()

## 2. Can this notebook see conda?

`shutil.which` looks for a program on your `PATH`, which is what your terminal does when you type a command. The cell then asks conda two read-only questions from the module: where does it look for packages, and which environments exist. If conda is not on the `PATH` of this process, it says so instead of failing.

In [ ]:
conda = shutil.which("conda")
print("conda found at:", conda)

if conda:
    for args in (["config", "--show", "channels"], ["env", "list"]):
        result = subprocess.run([conda, *args], capture_output=True, text=True, timeout=60)
        print("\n$ conda", " ".join(args))
        print(result.stdout.strip() or result.stderr.strip())
else:
    print("conda is not on this process's PATH. That is normal in some editors; your terminal may still find it.")

Look at the channel list. For this course it should contain `conda-forge`, and it should not rely on Anaconda's `defaults` channels (see *Conda configuration* in the module).

## 3. A venv is just a folder

The module introduces `venv` so that you recognize it. This cell builds one in the scratch folder (without pip, so it takes a second) and looks inside. Notice that it sits **inside your project folder**, which is why a venv must be listed in `.gitignore`. A conda environment lives elsewhere and cannot be committed by accident.

In [ ]:
import venv

project = SCRATCH / "demo-project"
project.mkdir()
venv.EnvBuilder(with_pip=False).create(project / ".venv")

print("Inside demo-project:", sorted(p.name for p in project.iterdir()))
print("Inside .venv       :", sorted(p.name for p in (project / ".venv").iterdir()))
print()
print((project / ".venv" / "pyvenv.cfg").read_text())      # which Python created it

## 4. Reading an `environment.yml`

`environment.yml` names the environment, lists the channels packages come from, and fixes the Python version. The function below is a very small reader for the simple files this course uses, and it uses only the standard library. It also warns about the two mistakes the module describes: no channels listed, and the `defaults` channel.

In [ ]:
def read_environment_yml(path):
    """Return (name, channels, python_pin, other_dependencies) from a simple environment.yml."""
    name, channels, python_pin, others = None, [], None, []
    section = None
    for raw in Path(path).read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if line.startswith("name:"):
            name = line.split(":", 1)[1].strip()
        elif line.startswith("channels:"):
            section = "channels"
        elif line.startswith("dependencies:"):
            section = "dependencies"
        elif line.startswith("- ") and section == "channels":
            channels.append(line[2:].strip())
        elif line.startswith("- ") and section == "dependencies":
            item = line[2:].strip()
            if item.startswith("python="):
                python_pin = item.split("=", 1)[1]
            else:
                others.append(item)
    return name, channels, python_pin, others


def review_environment_yml(path):
    name, channels, python_pin, others = read_environment_yml(path)
    print(f"name     : {name}")
    print(f"channels : {channels}")
    print(f"python   : {python_pin}")
    print(f"others   : {others}")
    if not channels:
        print("WARNING: no channels listed. Another machine will use its own settings.")
    if "defaults" in channels:
        print("WARNING: 'defaults' may trigger Anaconda's Terms of Service prompt. Use conda-forge.")
    if "conda-forge" in channels and "defaults" not in channels:
        print("Looks good: conda-forge, and no defaults channel.")


good = SCRATCH / "good_environment.yml"
good.write_text("name: module01-tool\nchannels:\n  - conda-forge\ndependencies:\n  - python=3.11\n  - pip\n")
review_environment_yml(good)

**Predict first:** what will the review say about this file, written the way many older guides do it?

In [ ]:
older = SCRATCH / "older_environment.yml"
older.write_text("name: module01-tool\nchannels:\n  - defaults\ndependencies:\n  - python=3.11\n  - pip\n")
review_environment_yml(older)

print()
nochannels = SCRATCH / "nochannels_environment.yml"
nochannels.write_text("name: module01-tool\ndependencies:\n  - python=3.11\n")
review_environment_yml(nochannels)

## 5. Are the pinned versions what is installed?

`requirements.txt` pins exact pip packages (`package==version`). This function compares each pin with what this environment actually has, which is the idea behind "someone else can rebuild exactly what I tested".

In [ ]:
from importlib import metadata


def check_requirements(path):
    """Compare each pinned line of a requirements file with the installed version."""
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        if "==" not in line:
            print(f"UNPINNED  {line}  (no == version, so installs can drift)")
            continue
        name, wanted = line.split("==")
        try:
            installed = metadata.version(name)
        except metadata.PackageNotFoundError:
            print(f"MISSING   {name}  (pinned {wanted}, not installed)")
            continue
        status = "OK" if installed == wanted else "MISMATCH"
        print(f"{status:<9} {name}  (pinned {wanted}, installed {installed})")


demo = SCRATCH / "requirements.txt"
demo.write_text(
    "# demo file\n"
    f"pip=={metadata.version('pip')}\n"
    "requests==0.0.1\n"
    "definitely-not-a-real-package==1.0\n"
    "pandas\n"
)
check_requirements(demo)

Four lines, four different outcomes: a pin that matches, a pin that does not, a package that is not installed, and a line with no pin at all.

---
## Try it

Create your own `module01-tool` environment (`conda create -n module01-tool python=3.11 ipykernel`), select it as this notebook's kernel, and run the notebook again. `describe_environment()` should now say *Good: working in the 'module01-tool' environment.*

Then write your own `environment.yml` in the next cell and review it with `review_environment_yml`.

In [ ]:
# Write your environment.yml text here, save it with Path(...).write_text(...), then review it.

## Clean up

Removes the scratch folder.

In [ ]:
shutil.rmtree(SCRATCH)
print("Scratch folder removed:", not SCRATCH.exists())